In [1]:
%run shared_imports.py
from decimal import Decimal
from dataclasses import dataclass
from sqlalchemy.orm.query import Query


In [2]:
engine = make_engine("settings.toml")
session = Session(engine)


In [12]:
query = session.query(Feedback.key_name, Feedback.json).join(Round).filter(
    Feedback.key_name.in_(["ai_controlled_elite_loss", "player_controlled_elite_loss", "player_controlled_elite_win", "ai_controlled_elite_win"]),
    Round.start_datetime >= datetime(2023, 10, 2),
    Round.start_datetime < datetime(2024, 10, 2))

raw_elite_winloss = pd.read_sql_query(query.statement, session.connection())

In [13]:
raw_elite_winloss

,key_name,json
0,ai_controlled_elite_loss,{'data': {'pandora': 2}}
1,ai_controlled_elite_win,{'data': {'pandora': 2}}
2,ai_controlled_elite_loss,{'data': {'herald': 1}}
3,ai_controlled_elite_win,{'data': {'legionnaire': 1}}
4,player_controlled_elite_win,{'data': {'pandora': 1}}
...,...,...
555,ai_controlled_elite_win,{'data': {'goliath broodmother': 1}}
556,ai_controlled_elite_win,{'data': {'pandora': 1}}
557,ai_controlled_elite_win,{'data': {'legionnaire': 1}}
558,ai_controlled_elite_loss,{'data': {'pandora': 1}}


In [14]:
def apply_elite_winloss(text):
    l = text['data']
    result = list()
    for k, v in l.items():
        result.append((k, v))
    if len(result):
        keys, values = zip(*result)
        series = pd.Series(values, index=keys)
        # print(series)
        return series
    else:
        return pd.Series()

def get_elite_winloss(df):
    return pd.concat([df, df['json'].apply(apply_elite_winloss)], axis=1).drop(['json'], axis=1)

In [15]:
elite_winloss = get_elite_winloss(raw_elite_winloss)

In [16]:
elite_winloss

,key_name,pandora,herald,legionnaire,goliath broodmother
0,ai_controlled_elite_loss,2.0,NaN,NaN,NaN
1,ai_controlled_elite_win,2.0,NaN,NaN,NaN
2,ai_controlled_elite_loss,NaN,1.0,NaN,NaN
3,ai_controlled_elite_win,NaN,NaN,1.0,NaN
4,player_controlled_elite_win,1.0,NaN,NaN,NaN
...,...,...,...,...,...
555,ai_controlled_elite_win,NaN,NaN,NaN,1.0
556,ai_controlled_elite_win,1.0,NaN,NaN,NaN
557,ai_controlled_elite_win,NaN,NaN,1.0,NaN
558,ai_controlled_elite_loss,1.0,NaN,NaN,NaN


In [19]:
elite_winloss.groupby(['key_name']).sum().astype(int)

,pandora,herald,legionnaire,goliath broodmother
key_name,,,,
ai_controlled_elite_loss,33,48,32,34
ai_controlled_elite_win,56,59,73,68
player_controlled_elite_loss,39,40,29,30
player_controlled_elite_win,19,17,25,40


In [261]:
totals_df.drop(['datetime'], inplace=True, axis=1)

In [268]:
totals_df[list(parts_kits_names)].sum().astype(int).sort_values()

Arc Revolver                  33
Energy Crossbow               47
u-ION Silencer                77
Plasma Pistol                102
Temperature Gun              103
LWAP Laser Sniper            107
Decloner                     109
SPRK-12 Pistol               152
Accelerator Laser Cannon     303
Ion Carbine                  445
Xray Laser Gun               635
Immolator Laser Gun          756
Advanced Energy Gun         1587
dtype: int32